# FineWeb GPT Trainer (Notebook)

Interactive wrapper around `train_fineweb_gpt.py` with preset + CLI-style args support.

In [ ]:
from pathlib import Path
import importlib.util
import sys, shlex

SCRIPT_PATH = Path("train_fineweb_gpt.py").resolve()
assert SCRIPT_PATH.exists(), f"Missing script: {SCRIPT_PATH}"

spec = importlib.util.spec_from_file_location("fineweb_train", SCRIPT_PATH)
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

print("Loaded:", SCRIPT_PATH)
print("Presets:", list(train_mod.PRESETS.keys()))


In [ ]:
# Optional UI controls (works in JupyterLab/Notebook)
try:
    import ipywidgets as widgets
    from IPython.display import display
    presets = sorted(getattr(train_mod, "PRESETS", {}).keys())
    if not presets:
        print("Warning: train_mod.PRESETS missing. Run the first cell again to reload train_fineweb_gpt.py")
    preset_dd = widgets.Dropdown(
        options=[("custom", "")] + [(k, k) for k in presets],
        value="",
        description="Preset:",
    )
    cli_tb = widgets.Text(
        value="--config CC-MAIN-2025-26",
        description="CLI args:",
        layout=widgets.Layout(width="80%"),
    )
    display(preset_dd, cli_tb)
    print("Use preset_dd.value and cli_tb.value in the next cell.")
except Exception as e:
    print("ipywidgets unavailable; use plain variables in next cell.")
    print(e)


In [ ]:
# Build args exactly like CLI parsing in train_fineweb_gpt.py

def build_args(preset="", cli_args=""):
    argv = ["train_fineweb_gpt.py"]
    if preset:
        argv += ["--preset", preset]
    if cli_args.strip():
        argv += shlex.split(cli_args)

    prev_argv = sys.argv
    try:
        sys.argv = argv
        args = train_mod.parse_args()
    finally:
        sys.argv = prev_argv
    return args

# If widgets exist, use them; otherwise edit these directly:
preset = preset_dd.value if 'preset_dd' in globals() else ""
cli_args = cli_tb.value if 'cli_tb' in globals() else "--config CC-MAIN-2025-26 --train-steps 2000"

args = build_args(preset=preset, cli_args=cli_args)
print("Resolved args:")
for k, v in vars(args).items():
    print(f"  {k}: {v}")


In [ ]:
# Pipeline runner: separate tokenizer build and full training
# - MODE="tokenizer" builds tokenizer only (CPU-safe)
# - MODE="train" runs full training loop

MODE = "tokenizer"  # one of: tokenizer, train

# Optional override: only used when MODE == "train"
RUN_TRAINING = True

import copy

run_args = copy.deepcopy(args)

if MODE == "tokenizer":
    # Force tokenizer build path only; avoid GPU model allocation.
    tok_path = run_args.tokenizer_model
    try:
        sp = train_mod.spm.SentencePieceProcessor(model_file=tok_path)
        print(f"Tokenizer already exists: {tok_path} | vocab={sp.vocab_size()}")
    except Exception:
        print(f"Building tokenizer -> {tok_path} using seed_docs={run_args.seed_docs}")
        sp = train_mod.ensure_tokenizer(run_args)
        print(f"Built tokenizer: {tok_path} | vocab={sp.vocab_size()}")

elif MODE == "train":
    if not RUN_TRAINING:
        print("Set RUN_TRAINING = True to start training.")
    else:
        device = "cuda" if train_mod.torch.cuda.is_available() else "cpu"
        if device == "cuda":
            train_mod.torch.set_float32_matmul_precision("high")
            train_mod.torch.backends.cuda.matmul.allow_tf32 = True
            train_mod.torch.backends.cudnn.allow_tf32 = True

        sp = train_mod.ensure_tokenizer(run_args)
        vocab = sp.vocab_size()

        train_stream = train_mod.StreamingBatcher(
            sp, run_args.config, run_args.context, run_args.batch_size,
            queue_size=run_args.queue_size, num_workers=run_args.num_workers
        )
        val_stream = train_mod.StreamingBatcher(
            sp, run_args.config, run_args.context, run_args.batch_size,
            queue_size=max(16, run_args.queue_size // 2), num_workers=max(1, run_args.num_workers // 2)
        )

        model = train_mod.GPT(vocab, run_args.context, run_args.n_embd, run_args.n_head, run_args.n_layer, run_args.dropout).to(device)
        if device == "cuda" and train_mod.platform.system() != "Windows":
            try:
                model = train_mod.torch.compile(model)
            except Exception:
                pass

        opt = None
        if device == "cuda":
            try:
                print("Optimizing with AdamW fused")
                opt = train_mod.torch.optim.AdamW(
                    model.parameters(), lr=run_args.lr, betas=(0.9, 0.95),
                    weight_decay=run_args.weight_decay, fused=True,
                )
            except (TypeError, RuntimeError) as e:
                print(f"Encountered Error '{e}', Falling back to AdamW foreach")
                try:
                    opt = train_mod.torch.optim.AdamW(
                        model.parameters(), lr=run_args.lr, betas=(0.9, 0.95),
                        weight_decay=run_args.weight_decay, foreach=True,
                    )
                except (TypeError, RuntimeError):
                    print("AdamW optimization failed; using plain AdamW")

        if opt is None:
            opt = train_mod.torch.optim.AdamW(
                model.parameters(), lr=run_args.lr, betas=(0.9, 0.95), weight_decay=run_args.weight_decay
            )

        scaler = train_mod.torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

        def get_lr(step):
            if step < run_args.warmup_steps:
                return run_args.lr * (step + 1) / max(run_args.warmup_steps, 1)
            progress = (step - run_args.warmup_steps) / max(run_args.train_steps - run_args.warmup_steps, 1)
            progress = min(max(progress, 0.0), 1.0)
            cosine = 0.5 * (1.0 + train_mod.math.cos(train_mod.math.pi * progress))
            return run_args.min_lr + cosine * (run_args.lr - run_args.min_lr)

        @train_mod.torch.no_grad()
        def eval_loss(iters):
            model.eval()
            vals = []
            for _ in range(iters):
                xb, yb = val_stream.next(device)
                with train_mod.torch.amp.autocast("cuda", enabled=(device == "cuda")):
                    _, loss = model(xb, yb)
                vals.append(loss.item())
            model.train()
            return sum(vals) / len(vals)

        params = sum(p.numel() for p in model.parameters())
        est = train_mod.estimate_params(vocab, run_args.context, run_args.n_embd, run_args.n_layer)
        print(
            f"device={device} | mode=train | preset={run_args.preset or 'custom'} | vocab={vocab} | params={params:,} "
            f"(est {est:,}) | config={run_args.config} | grad_accum={run_args.grad_accum} | workers={run_args.num_workers}"
        )

        ckpt_path = "fineweb_gpt.ckpt"
        start_time = train_mod.time.perf_counter()
        last_step_dt = 0.0
        tokens_per_step = run_args.batch_size * run_args.context * run_args.grad_accum

        try:
            for step in range(run_args.train_steps + 1):
                now = train_mod.time.perf_counter()
                elapsed = now - start_time
                avg_step = elapsed / max(step, 1)
                eta = max(run_args.train_steps - step, 0) * avg_step

                if step == 1 or step % run_args.eval_every == 0:
                    eval_start = train_mod.time.perf_counter()
                    v = eval_loss(run_args.eval_iters)
                    eval_dt = train_mod.time.perf_counter() - eval_start
                    cur_lr = opt.param_groups[0]["lr"]
                    toks_per_s = (tokens_per_step / last_step_dt) if last_step_dt > 0 else 0.0
                    print(
                        f"step {step:5d} | val {v:.4f} | ppl {train_mod.math.exp(v):.2f} | "
                        f"lr {cur_lr:.2e} | dt {last_step_dt:.2f}s | tok/s {toks_per_s:,.0f} | "
                        f"eval {eval_dt:.2f}s | elapsed {elapsed/60:.1f}m | eta {eta/60:.1f}m"
                    )

                step_start = train_mod.time.perf_counter()
                cur_lr = get_lr(step)
                for pg in opt.param_groups:
                    pg["lr"] = cur_lr
                opt.zero_grad(set_to_none=True)
                for _ in range(run_args.grad_accum):
                    xb, yb = train_stream.next(device)
                    with train_mod.torch.amp.autocast("cuda", enabled=(device == "cuda")):
                        _, loss = model(xb, yb)
                        loss = loss / run_args.grad_accum
                    scaler.scale(loss).backward()

                scaler.unscale_(opt)
                train_mod.torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                last_step_dt = train_mod.time.perf_counter() - step_start

                if step > 0 and step % run_args.ckpt_every == 0:
                    sd = model._orig_mod.state_dict() if hasattr(model, "_orig_mod") else model.state_dict()
                    train_mod.torch.save({"state_dict": sd, "args": vars(run_args), "vocab": vocab}, ckpt_path)
                    print(f"checkpoint -> {ckpt_path}")

            sd = model._orig_mod.state_dict() if hasattr(model, "_orig_mod") else model.state_dict()
            train_mod.torch.save({"state_dict": sd, "args": vars(run_args), "vocab": vocab}, ckpt_path)
            print(f"saved -> {ckpt_path}")
        finally:
            train_stream.close()
            val_stream.close()
else:
    raise ValueError("MODE must be one of: tokenizer, train")
